In [1]:
# -*- coding: utf-8 -*-
"""
Validation Step 3 — conclusion-robustness checks
V20-aligned script for the ICST/MSR emulator-testing study.

What this version fixes
-----------------------
1) Uses the strict executed-workload signature basis already adopted in the paper:
   - study_signature_hash
   - study_runner_os_bucket
   - study_job_count_exec_bucket
   - study_step_count_exec_bucket

2) Screens exact signatures inside the Base timing regime for timing-focused Step 3 checks.
   Eligibility rule:
   - total >= 80
   - at least 2 styles with >= 15 records each

3) Defines Tier 2 per eligible exact signature, not as one pooled family.
   For each exact signature, Tier 2 is built by:
   - holding runner OS bucket fixed
   - broadening executed job-count bucket to adjacent categories
   - broadening executed step-count bucket to adjacent categories

4) Produces both exact-signature and paired-family outputs for Step 3.

Outputs
-------
Creates a folder with:
- step3_regime_summary.csv
- step3_regime_ordering_vs_base.csv
- step3_signature_candidates.csv
- step3_signature_family_definitions.csv
- step3_exact_signature_summary.csv
- step3_exact_signature_conclusion_stability.csv
- step3_family_summary.csv
- step3_family_conclusion_stability.csv
- step3_notes.txt
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_MAIN = BASE_DIR / "MainDataset.csv"
OUT_DIR = BASE_DIR / r"0.2-Validation\Step 3 - Robustness_Check"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_SIGNATURE_TOTAL_N = 80
MIN_SIGNATURE_STYLE_N = 15
MIN_SIGNATURE_USABLE_STYLES = 2

STYLE_ORDER_ALL = ["Community", "Third-Party", "GMD", "Custom"]
USABLE_VERDICTS = {"success", "failure"}

LAYER1_MEASURES = {
    "run_duration": "study_run_duration_seconds",
    "l1_time_to_instr_env": "study_layer1_time_to_instrumentation_envelope_seconds",
    "l1_instr_job_env": "study_layer1_instrumentation_job_envelope_seconds",
    "l1_post_instr_tail": "study_layer1_post_instrumentation_tail_seconds",
}

LAYER2_MEASURES = {
    "l2_pre_invocation": "study_pre_invocation_selected_stage3_seconds",
    "l2_execution_window": "study_invocation_execution_window_selected_stage3_seconds",
    "l2_post_invocation": "study_post_invocation_selected_stage3_seconds",
}

SIG_HASH_COL = "study_signature_hash"
RUNNER_OS_COL = "study_runner_os_bucket"
JOB_BUCKET_COL = "study_job_count_exec_bucket"
STEP_BUCKET_COL = "study_step_count_exec_bucket"
COARSENED_PREFIX = "coarsened_family__"

JOB_BUCKET_ADJ = {
    "1": {"1", "2_3"},
    "2_3": {"1", "2_3", "4_6"},
    "4_6": {"2_3", "4_6", ">6"},
    ">6": {"4_6", ">6"},
}

STEP_BUCKET_ADJ = {
    "<=20": {"<=20", "21_40"},
    "21_40": {"<=20", "21_40", "41_80"},
    "41_80": {"21_40", "41_80", ">80"},
    ">80": {"41_80", ">80"},
}

# ============================================================
# HELPERS
# ============================================================
def norm_bool(series: pd.Series) -> pd.Series:
    s = series.copy()
    if pd.api.types.is_bool_dtype(s):
        return s.astype("boolean")
    s = s.replace({1: True, 0: False})
    s = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
    }
    return s.map(mapping).astype("boolean")


def to_num(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def pct95(series: pd.Series) -> float:
    s = series.dropna()
    if s.empty:
        return np.nan
    return float(np.percentile(s, 95))


def iqr(series: pd.Series) -> float:
    s = series.dropna()
    if s.empty:
        return np.nan
    return float(np.percentile(s, 75) - np.percentile(s, 25))


def safe_ratio(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return float(a / b)


def normalize_bucket_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    return s if s else np.nan


def add_measure_summary(rows, df_part: pd.DataFrame, regime_name: str, layer_name: str, measures: dict):
    for measure_key, col in measures.items():
        if col not in df_part.columns:
            continue
        for style, g in df_part.groupby("style", dropna=False):
            s = to_num(g[col])
            n_total = len(g)
            n_non_missing = int(s.notna().sum())
            med = float(s.median()) if n_non_missing else np.nan
            rows.append({
                "regime": regime_name,
                "layer": layer_name,
                "measure": measure_key,
                "column_name": col,
                "style": style,
                "n_total_records": n_total,
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, n_total),
                "median": med,
                "q1": float(s.quantile(0.25)) if n_non_missing else np.nan,
                "q3": float(s.quantile(0.75)) if n_non_missing else np.nan,
                "iqr": iqr(s) if n_non_missing else np.nan,
                "p95": pct95(s) if n_non_missing else np.nan,
            })


def median_style_order(summary_df: pd.DataFrame, regime: str, measure_key: str, min_n: int = 10) -> list:
    part = summary_df[
        (summary_df["regime"] == regime)
        & (summary_df["measure"] == measure_key)
        & (summary_df["n_non_missing"] >= min_n)
    ].copy()
    if part.empty:
        return []
    part["style_rank"] = part["style"].map({s: i for i, s in enumerate(STYLE_ORDER_ALL)})
    part = part.sort_values(["median", "style_rank"], na_position="last")
    return part["style"].tolist()


def compare_orderings(reference_order: list, other_order: list) -> str:
    if len(other_order) < 2:
        return "not_testable"
    if other_order == reference_order:
        return "preserved"
    if len(reference_order) >= 1 and len(other_order) >= 1 and reference_order[0] == other_order[0]:
        return "partially_preserved"
    if len(reference_order) >= 2 and len(other_order) >= 2 and set(reference_order[:2]) == set(other_order[:2]):
        return "partially_preserved"
    return "changed"


def get_family_mask(df: pd.DataFrame, sig: str, runner_os: str, job_bucket: str, step_bucket: str) -> pd.Series:
    precomputed_col = f"{COARSENED_PREFIX}{sig}"
    if precomputed_col in df.columns:
        try:
            return norm_bool(df[precomputed_col]).fillna(False)
        except Exception:
            pass
    return (
        df[SIG_HASH_COL].notna()
        & df[RUNNER_OS_COL].eq(runner_os)
        & df[JOB_BUCKET_COL].isin(JOB_BUCKET_ADJ.get(job_bucket, {job_bucket}))
        & df[STEP_BUCKET_COL].isin(STEP_BUCKET_ADJ.get(step_bucket, {step_bucket}))
    )

# ============================================================
# LOAD + NORMALIZE
# ============================================================
if not IN_MAIN.exists():
    raise FileNotFoundError(f"MainDataset.csv not found at: {IN_MAIN}")

df = pd.read_csv(IN_MAIN, low_memory=False)

for c in ["Base", "Robust", "controller_attempt_eq_1", "controller_style_in_scope", "controller_instru_job_count_gt0"]:
    if c in df.columns:
        df[c] = norm_bool(df[c])

if "run_attempt" in df.columns:
    df["run_attempt"] = pd.to_numeric(df["run_attempt"], errors="coerce")

for col in list(LAYER1_MEASURES.values()) + list(LAYER2_MEASURES.values()):
    if col in df.columns:
        df[col] = to_num(df[col])

if "study_layer2_measurement_mode" in df.columns:
    df["layer2_observable"] = df["study_layer2_measurement_mode"].astype(str).str.strip().eq("measured_step_based")
else:
    first_l2 = next(iter(LAYER2_MEASURES.values()))
    df["layer2_observable"] = df[first_l2].notna() if first_l2 in df.columns else False

df["controller_usable_verdict"] = df["run_conclusion"].astype(str).str.strip().str.lower().isin(USABLE_VERDICTS)

for c in [SIG_HASH_COL, RUNNER_OS_COL, JOB_BUCKET_COL, STEP_BUCKET_COL]:
    if c not in df.columns:
        raise ValueError(f"Missing required signature field: {c}")
    df[c] = df[c].map(normalize_bucket_value)

# ============================================================
# DEFINE REGIMES
# ============================================================
all_run_per_style_mask = pd.Series(True, index=df.index)
if "controller_style_in_scope" in df.columns:
    all_run_per_style_mask &= df["controller_style_in_scope"].fillna(False)
if "controller_instru_job_count_gt0" in df.columns:
    all_run_per_style_mask &= df["controller_instru_job_count_gt0"].fillna(False)

attempt_eq_1_mask = all_run_per_style_mask.copy()
if "controller_attempt_eq_1" in df.columns:
    attempt_eq_1_mask &= df["controller_attempt_eq_1"].fillna(False)

controlled_subset_mask = (
    df["Base"].fillna(False)
    if "Base" in df.columns
    else (all_run_per_style_mask & df["controller_attempt_eq_1"].fillna(False) & df["controller_usable_verdict"].fillna(False))
)

rerun_usable_verdict_mask = all_run_per_style_mask & df["controller_usable_verdict"].fillna(False) & (df["run_attempt"].fillna(1) > 1)

regimes = {
    "all_run_per_style": all_run_per_style_mask,
    "attempt_eq_1": attempt_eq_1_mask,
    "controlled_subset": controlled_subset_mask,
    "rerun_usable_verdict": rerun_usable_verdict_mask,
}
REFERENCE_REGIME = "controlled_subset"

# ============================================================
# CHECK A — CONTROLLER-REGIME ROBUSTNESS
# ============================================================
summary_rows = []
for regime_name, mask in regimes.items():
    part = df.loc[mask].copy()
    if part.empty:
        continue
    add_measure_summary(summary_rows, part, regime_name, "layer1", LAYER1_MEASURES)
    add_measure_summary(summary_rows, part.loc[part["layer2_observable"]].copy(), regime_name, "layer2", LAYER2_MEASURES)

regime_summary = pd.DataFrame(summary_rows)
regime_summary.to_csv(OUT_DIR / "step3_regime_summary.csv", index=False)

ordering_rows = []
for layer_name, measures in [("layer1", LAYER1_MEASURES), ("layer2", LAYER2_MEASURES)]:
    for measure_key in measures.keys():
        ref_order = median_style_order(regime_summary, REFERENCE_REGIME, measure_key, min_n=10)
        for regime_name in regimes.keys():
            tgt_order = median_style_order(regime_summary, regime_name, measure_key, min_n=10)
            ordering_rows.append({
                "layer": layer_name,
                "measure": measure_key,
                "reference_regime": REFERENCE_REGIME,
                "target_regime": regime_name,
                "reference_order": " > ".join(ref_order) if ref_order else "",
                "target_order": " > ".join(tgt_order) if tgt_order else "",
                "ordering_status_vs_reference": compare_orderings(ref_order, tgt_order),
            })

pd.DataFrame(ordering_rows).to_csv(OUT_DIR / "step3_regime_ordering_vs_base.csv", index=False)

# ============================================================
# CHECK B — EXACT SIGNATURES + PAIRED TIER 2 FAMILIES
# ============================================================
sig_pool = df.loc[
    controlled_subset_mask
    & df[SIG_HASH_COL].notna()
    & df[RUNNER_OS_COL].notna()
    & df[JOB_BUCKET_COL].notna()
    & df[STEP_BUCKET_COL].notna()
].copy()

sig_counts = (
    sig_pool.groupby([SIG_HASH_COL, "style"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
)

sig_wide = sig_counts.pivot_table(index=SIG_HASH_COL, columns="style", values="n", fill_value=0).reset_index()
for s in STYLE_ORDER_ALL:
    if s not in sig_wide.columns:
        sig_wide[s] = 0
sig_wide["total_n"] = sig_wide[STYLE_ORDER_ALL].sum(axis=1)
sig_wide["usable_style_count"] = (sig_wide[STYLE_ORDER_ALL] >= MIN_SIGNATURE_STYLE_N).sum(axis=1)

sig_meta = sig_pool[[SIG_HASH_COL, RUNNER_OS_COL, JOB_BUCKET_COL, STEP_BUCKET_COL]].drop_duplicates(subset=[SIG_HASH_COL]).copy()
sig_candidates = sig_wide.merge(sig_meta, on=SIG_HASH_COL, how="left")
sig_candidates["eligible"] = (
    (sig_candidates["total_n"] >= MIN_SIGNATURE_TOTAL_N)
    & (sig_candidates["usable_style_count"] >= MIN_SIGNATURE_USABLE_STYLES)
)

family_rows = []
for _, row in sig_candidates.iterrows():
    sig = row[SIG_HASH_COL]
    family_mask_all = get_family_mask(df, sig, row[RUNNER_OS_COL], row[JOB_BUCKET_COL], row[STEP_BUCKET_COL])
    family_mask_base = family_mask_all & controlled_subset_mask
    family_rows.append({
        SIG_HASH_COL: sig,
        RUNNER_OS_COL: row[RUNNER_OS_COL],
        JOB_BUCKET_COL: row[JOB_BUCKET_COL],
        STEP_BUCKET_COL: row[STEP_BUCKET_COL],
        "eligible": bool(row["eligible"]),
        "exact_base_n": int(row["total_n"]),
        "family_base_n": int(family_mask_base.sum()),
        "family_all_n": int(family_mask_all.sum()),
        "job_bucket_family": "{" + ", ".join(sorted(JOB_BUCKET_ADJ.get(row[JOB_BUCKET_COL], {row[JOB_BUCKET_COL]}), key=lambda x: ["1", "2_3", "4_6", ">6"].index(x) if x in ["1", "2_3", "4_6", ">6"] else 99)) + "}",
        "step_bucket_family": "{" + ", ".join(sorted(STEP_BUCKET_ADJ.get(row[STEP_BUCKET_COL], {row[STEP_BUCKET_COL]}), key=lambda x: ["<=20", "21_40", "41_80", ">80"].index(x) if x in ["<=20", "21_40", "41_80", ">80"] else 99)) + "}",
    })

sig_family_defs = pd.DataFrame(family_rows)
sig_candidates = sig_candidates.merge(sig_family_defs.drop(columns=[RUNNER_OS_COL, JOB_BUCKET_COL, STEP_BUCKET_COL, "eligible"]), on=SIG_HASH_COL, how="left")
sig_candidates = sig_candidates.sort_values(["eligible", "total_n", "usable_style_count"], ascending=[False, False, False]).reset_index(drop=True)
sig_candidates.to_csv(OUT_DIR / "step3_signature_candidates.csv", index=False)
sig_family_defs.to_csv(OUT_DIR / "step3_signature_family_definitions.csv", index=False)

eligible_signatures = sig_candidates.loc[sig_candidates["eligible"], SIG_HASH_COL].tolist()

exact_rows = []
exact_stability_rows = []
family_rows_summary = []
family_stability_rows = []

for sig in eligible_signatures:
    row = sig_candidates.loc[sig_candidates[SIG_HASH_COL] == sig].iloc[0]
    exact_part = sig_pool.loc[sig_pool[SIG_HASH_COL] == sig].copy()
    family_mask_base = get_family_mask(df, sig, row[RUNNER_OS_COL], row[JOB_BUCKET_COL], row[STEP_BUCKET_COL]) & controlled_subset_mask
    family_part = df.loc[family_mask_base].copy()

    for style, g in exact_part.groupby("style", dropna=False):
        for measure_key, col in LAYER1_MEASURES.items():
            s = to_num(g[col]) if col in g.columns else pd.Series(dtype=float)
            n_non_missing = int(s.notna().sum())
            exact_rows.append({
                SIG_HASH_COL: sig,
                "tier": "exact",
                "layer": "layer1",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g)),
                "median": float(s.median()) if n_non_missing else np.nan,
                "iqr": iqr(s) if n_non_missing else np.nan,
                "p95": pct95(s) if n_non_missing else np.nan,
            })
        g2 = g.loc[g["layer2_observable"]].copy()
        for measure_key, col in LAYER2_MEASURES.items():
            s = to_num(g2[col]) if col in g2.columns else pd.Series(dtype=float)
            n_non_missing = int(s.notna().sum())
            exact_rows.append({
                SIG_HASH_COL: sig,
                "tier": "exact",
                "layer": "layer2",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g2),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g2)),
                "median": float(s.median()) if n_non_missing else np.nan,
                "iqr": iqr(s) if n_non_missing else np.nan,
                "p95": pct95(s) if n_non_missing else np.nan,
            })

    for style, g in family_part.groupby("style", dropna=False):
        for measure_key, col in LAYER1_MEASURES.items():
            s = to_num(g[col]) if col in g.columns else pd.Series(dtype=float)
            n_non_missing = int(s.notna().sum())
            family_rows_summary.append({
                SIG_HASH_COL: sig,
                "tier": "family",
                "layer": "layer1",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g)),
                "median": float(s.median()) if n_non_missing else np.nan,
                "iqr": iqr(s) if n_non_missing else np.nan,
                "p95": pct95(s) if n_non_missing else np.nan,
            })
        g2 = g.loc[g["layer2_observable"]].copy()
        for measure_key, col in LAYER2_MEASURES.items():
            s = to_num(g2[col]) if col in g2.columns else pd.Series(dtype=float)
            n_non_missing = int(s.notna().sum())
            family_rows_summary.append({
                SIG_HASH_COL: sig,
                "tier": "family",
                "layer": "layer2",
                "measure": measure_key,
                "style": style,
                "n_total_records": len(g2),
                "n_non_missing": n_non_missing,
                "coverage_rate": safe_ratio(n_non_missing, len(g2)),
                "median": float(s.median()) if n_non_missing else np.nan,
                "iqr": iqr(s) if n_non_missing else np.nan,
                "p95": pct95(s) if n_non_missing else np.nan,
            })

exact_summary = pd.DataFrame(exact_rows)
family_summary = pd.DataFrame(family_rows_summary)

for sig in eligible_signatures:
    exact_temp = exact_summary[exact_summary[SIG_HASH_COL] == sig].copy()
    family_temp = family_summary[family_summary[SIG_HASH_COL] == sig].copy()
    meta = sig_candidates.loc[sig_candidates[SIG_HASH_COL] == sig].iloc[0]

    for layer_name, measures in [("layer1", LAYER1_MEASURES), ("layer2", LAYER2_MEASURES)]:
        for measure_key in measures.keys():
            ref_order = median_style_order(regime_summary, REFERENCE_REGIME, measure_key, min_n=10)

            part_e = exact_temp[(exact_temp["layer"] == layer_name) & (exact_temp["measure"] == measure_key) & (exact_temp["n_non_missing"] >= MIN_SIGNATURE_STYLE_N)].copy()
            exact_order = []
            if not part_e.empty:
                part_e["style_rank"] = part_e["style"].map({s: i for i, s in enumerate(STYLE_ORDER_ALL)})
                part_e = part_e.sort_values(["median", "style_rank"])
                exact_order = part_e["style"].tolist()

            part_f = family_temp[(family_temp["layer"] == layer_name) & (family_temp["measure"] == measure_key) & (family_temp["n_non_missing"] >= MIN_SIGNATURE_STYLE_N)].copy()
            family_order = []
            if not part_f.empty:
                part_f["style_rank"] = part_f["style"].map({s: i for i, s in enumerate(STYLE_ORDER_ALL)})
                part_f = part_f.sort_values(["median", "style_rank"])
                family_order = part_f["style"].tolist()

            base_info = {
                SIG_HASH_COL: sig,
                "reference_regime": REFERENCE_REGIME,
                "layer": layer_name,
                "measure": measure_key,
                "reference_order": " > ".join(ref_order) if ref_order else "",
                "community_n": int(meta.get("Community", 0)),
                "third_party_n": int(meta.get("Third-Party", 0)),
                "gmd_n": int(meta.get("GMD", 0)),
                "custom_n": int(meta.get("Custom", 0)),
                RUNNER_OS_COL: meta.get(RUNNER_OS_COL, ""),
                JOB_BUCKET_COL: meta.get(JOB_BUCKET_COL, ""),
                STEP_BUCKET_COL: meta.get(STEP_BUCKET_COL, ""),
            }

            exact_stability_rows.append({
                **base_info,
                "tier": "exact",
                "target_order": " > ".join(exact_order) if exact_order else "",
                "ordering_status_vs_reference": compare_orderings(ref_order, exact_order),
                "target_total_n": int(meta.get("exact_base_n", meta.get("total_n", np.nan))),
            })

            family_stability_rows.append({
                **base_info,
                "tier": "family",
                "target_order": " > ".join(family_order) if family_order else "",
                "ordering_status_vs_reference": compare_orderings(ref_order, family_order),
                "target_total_n": int(meta.get("family_base_n", np.nan)),
            })

exact_summary.to_csv(OUT_DIR / "step3_exact_signature_summary.csv", index=False)
pd.DataFrame(exact_stability_rows).to_csv(OUT_DIR / "step3_exact_signature_conclusion_stability.csv", index=False)
family_summary.to_csv(OUT_DIR / "step3_family_summary.csv", index=False)
pd.DataFrame(family_stability_rows).to_csv(OUT_DIR / "step3_family_conclusion_stability.csv", index=False)

notes = []
notes.append("Validation Step 3 — automated conclusion-robustness outputs (V20-aligned)")
notes.append("")
notes.append(f"Input file: {IN_MAIN}")
notes.append(f"Output folder: {OUT_DIR}")
notes.append("")
notes.append("Method alignment:")
notes.append("- Base timing regime = instrumentation-executed first-attempt usable-verdict records.")
notes.append("- Exact signature = strict executed-workload fingerprint from runner OS + executed job bucket + executed step bucket.")
notes.append(f"- Eligibility = total >= {MIN_SIGNATURE_TOTAL_N} and at least {MIN_SIGNATURE_USABLE_STYLES} styles with >= {MIN_SIGNATURE_STYLE_N} records.")
notes.append("- Tier 2 = per-signature paired family with fixed runner OS and adjacent executed job/step buckets.")
notes.append("")
notes.append("Files produced:")
for fn in [
    "step3_regime_summary.csv",
    "step3_regime_ordering_vs_base.csv",
    "step3_signature_candidates.csv",
    "step3_signature_family_definitions.csv",
    "step3_exact_signature_summary.csv",
    "step3_exact_signature_conclusion_stability.csv",
    "step3_family_summary.csv",
    "step3_family_conclusion_stability.csv",
]:
    notes.append(f"- {fn}")
notes.append("")
notes.append("Interpretation guide:")
notes.append("1) step3_regime_ordering_vs_base.csv checks controller-regime robustness against Base.")
notes.append("2) step3_signature_candidates.csv reports exact-signature support and eligibility inside Base.")
notes.append("3) step3_signature_family_definitions.csv reports exact-to-family expansion for each signature.")
notes.append("4) step3_exact_signature_conclusion_stability.csv checks Tier 1 ordering stability vs Base.")
notes.append("5) step3_family_conclusion_stability.csv checks Tier 2 ordering stability vs Base.")
(OUT_DIR / "step3_notes.txt").write_text("\n".join(notes), encoding="utf-8")

print(f"Done. Outputs saved to: {OUT_DIR}")
print(f"Eligible signatures found: {eligible_signatures}")


Done. Outputs saved to: C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step 3 - Robustness_Check
Eligible signatures found: ['0bc0e933a2435166', '88f32b360855c277']
